# Reasoning Predictive Equity (RPE)

This notebook computes semantic alignment between two generations using either Sentence Transformers or a local TF-IDF cosine-similarity fallback.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def calculate_rpe(gen_0: str, gen_g: str, use_sbert: bool = False) -> float:
    """Compute RPE as cosine similarity between two generated responses.

    When ``use_sbert`` is true, Sentence Transformers is used if it is
    installed and the model can be loaded. Otherwise, the function
    falls back to a local TF-IDF representation.
    """
    if not isinstance(gen_0, str) or not isinstance(gen_g, str):
        raise TypeError("gen_0 and gen_g must both be strings")
    if not gen_0.strip() or not gen_g.strip():
        raise ValueError("gen_0 and gen_g must not be empty")

    if use_sbert:
        try:
            from sentence_transformers import SentenceTransformer

            model = SentenceTransformer("all-MiniLM-L6-v2")
            embeddings = model.encode([gen_0, gen_g])
            return float(cosine_similarity([embeddings[0]], [embeddings[1]])[0, 0])
        except (ImportError, OSError, RuntimeError) as exc:
            print(f"Sentence Transformers unavailable ({exc}). Falling back to TF-IDF.")

    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([gen_0, gen_g])
    return float(cosine_similarity(vectors[0], vectors[1])[0, 0])


## Empirical validation

The CSV is expected at `/mnt/data/rpe_vs_generation.csv`. The check is skipped with a clear message when the data file is not present.

In [ ]:
csv_path = Path("/mnt/data/rpe_vs_generation.csv")

if csv_path.exists():
    df_rpe = pd.read_csv(csv_path)
    required_column = "llama_70b"
    if required_column not in df_rpe.columns:
        raise KeyError(f"Expected column {required_column!r}; found {list(df_rpe.columns)!r}")
    values = pd.to_numeric(df_rpe[required_column], errors="coerce").dropna()
    if values.empty:
        raise ValueError(f"Column {required_column!r} contains no numeric values")
    print(f"Empirical RPE Range (Llama-70B): {values.min():.2f} to {values.max():.2f}")
else:
    print(f"Empirical validation skipped: {csv_path} was not found.")


## Example test case

In [ ]:
gen_0 = "The mathematical proof relies on the law of large numbers."
gen_5 = "Numbers are large in mathematics and proofs are important."
print(f"Calculated RPE (Gen 0 vs 5): {calculate_rpe(gen_0, gen_5):.4f}")
